# 🚀 Réentraînement & Benchmark NIDS HDVR : v4 → v5 (Google Colab)

**Configuration conseillée :** Exécution → Modifier le type d'exécution → **GPU (T4)** (gratuit sur Colab).
Ce notebook s'exécute à 100% sur Google Colab sans solliciter les ressources de votre machine locale.

**Fichiers attendus dans votre Google Drive sous `MyDrive/nids/` :**
- `classes.json`, `feature_cols.json` (les 58 features canoniques)
- `dnn_v4.pt`, `scaler_v4.joblib`, `meta_v4.json`, `env_v4.json` (modèle v4 de référence)
- Dataset d'entraînement sous `MyDrive/nids/data/` (ex: `cicids2017_nfstream.parquet` ou `.csv`)
- *(Optionnel)* `feedback_flows.json` ou `.csv` exporté depuis le dashboard SOC pour l'Active Learning

**Objectifs de ce notebook :**
1. Évaluer la baseline du modèle `v4` actuel sur les données de test (en gérant proprement l'entrée 59d).
2. Ré-entraîner `v5` sur les **58 caractéristiques strictes** (sans duplication du protocole).
3. Entraîner et benchmarker un modèle tabulaire de pointe (**LightGBM**) sur le même split.
4. Calibrer les probabilités (**Temperature Scaling**) pour réduire les fausses alertes.
5. *(Optionnel)* Fusionner les qualifications d'analystes SOC (`feedback_flows`).
6. Sauvegarder les artefacts `v5` (`dnn_v5.pt`, `scaler_v5.joblib`, `meta_v5.json`, `manifest_v5.json`) prêts à être rapatriés.


In [ ]:
# 1. Montage de Google Drive et installation des dépendances
from google.colab import drive
drive.mount('/content/drive')

!pip install -q lightgbm


## 1. Paramètres & Chemins


In [ ]:
BASE      = '/content/drive/MyDrive/nids'
DATA_PATH = f'{BASE}/data/cicids2017_nfstream.parquet'   # .parquet ou .csv
FEEDBACK_PATH = f'{BASE}/feedback_flows.json'            # optionnel (Active Learning)
LABEL_COL = 'label'
TIME_COL  = 'bidirectional_first_seen_ms'                # temps en millisecondes
LABEL_MAP = {}   # ex : {'DoS slowloris': 'DoS-slowloris', 'Web Attack': 'WebAttack'}

OUT_VERSION       = 'v5'
SEEDS             = [0, 1, 2]
CAP_TRAIN         = 40000      # plafond par classe (comme v4)
USE_CLASS_WEIGHTS = False      # v4 : aucun. Mettre True pour tester des poids 1/sqrt(n)
DROPOUT           = 0.3        # architecture v4 (dropout 0.3)
LR, BATCH, MAX_EPOCHS, PATIENCE = 1e-3, 1024, 60, 8


## 2. Chargement des métadonnées et du dataset


In [ ]:
import json, sys, os, joblib, numpy as np, pandas as pd, torch, torch.nn as nn, sklearn
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, classification_report, confusion_matrix

CLASSES  = json.load(open(f'{BASE}/classes.json'))
FEATURES = json.load(open(f'{BASE}/feature_cols.json'))
META_V4  = json.load(open(f'{BASE}/meta_v4.json'))
cls2id   = {c: i for i, c in enumerate(CLASSES)}
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device :', dev, '| classes :', len(CLASSES), '| features :', len(FEATURES))

# Versions : comparer avec env_v4.json
print('Actuel :', dict(python=sys.version.split()[0], torch=torch.__version__,
                       sklearn=sklearn.__version__, numpy=np.__version__, pandas=pd.__version__))
if os.path.exists(f'{BASE}/env_v4.json'):
    print('env_v4 :', json.load(open(f'{BASE}/env_v4.json')))

df = pd.read_parquet(DATA_PATH) if DATA_PATH.endswith('parquet') else pd.read_csv(DATA_PATH)
df[LABEL_COL] = df[LABEL_COL].astype(str).str.strip().replace(LABEL_MAP)
print('\nÉtiquettes hors des 12 classes (exclues) :')
print(df.loc[~df[LABEL_COL].isin(CLASSES), LABEL_COL].value_counts().to_string() or 'aucune')

# Détection automatique de la colonne temporelle
if TIME_COL not in df.columns:
    candidates = ['start', 'bidirectional_first_seen_ms', 'first_seen_ms', 'timestamp', 'time']
    found = [c for c in candidates if c in df.columns]
    if found:
        print(f"\n💡 INFO : '{TIME_COL}' absent, utilisation automatique de : '{found[0]}'")
        TIME_COL = found[0]
    else:
        raise ValueError(f"Colonne temporelle introuvable. Colonnes présentes : {list(df.columns)}")

missing = [c for c in FEATURES if c not in df.columns]
assert not missing, f'Colonnes absentes du dataset : {missing}'
df = df[df[LABEL_COL].isin(CLASSES)].dropna(subset=[TIME_COL]).reset_index(drop=True)
print('\nFlux conservés :', len(df))
print(df[LABEL_COL].value_counts().to_string())


## 3. Split chronologique anti-leakage (identique à v4)
Attaques : coupe chronologique 60/20/20 par classe, avec marge de 1 % entre les parties. Normal : blocs de 10 min tirés au hasard.


In [ ]:
rng = np.random.RandomState(0)

# Conversion robuste en timestamp numérique (millisecondes)
if pd.api.types.is_datetime64_any_dtype(df[TIME_COL]):
    t = (df[TIME_COL].astype('int64') // 10**6).values.astype(np.float64)
elif pd.api.types.is_numeric_dtype(df[TIME_COL]):
    t = df[TIME_COL].values.astype(np.float64)
else:
    t = (pd.to_datetime(df[TIME_COL]).astype('int64') // 10**6).values.astype(np.float64)

lab = df[LABEL_COL].values
y_all = np.array([cls2id[l] for l in lab])
split = np.full(len(df), '', dtype=object)

# Si le split existe déjà dans le dataset (créé par NIDS.ipynb), l'utiliser directement
if 'split' in df.columns and set(df['split'].dropna().unique()).issuperset({'train', 'val', 'test'}):
    print("💡 Colonne 'split' pré-calculée trouvée dans le dataset !")
    split = df['split'].values
else:
    for c in CLASSES:
        idx = np.where(lab == c)[0]
        if c == 'BENIGN':
            blocks = (t[idx] // 600000).astype(np.int64)         # blocs de 10 min
            ub = np.unique(blocks); rng.shuffle(ub)
            n = len(ub); a, b = int(.6 * n), int(.8 * n)
            m = {k: 'train' for k in ub[:a]}
            m.update({k: 'val' for k in ub[a:b]}); m.update({k: 'test' for k in ub[b:]})
            split[idx] = [m[k] for k in blocks]
        else:
            o = idx[np.argsort(t[idx], kind='stable')]
            n = len(o)
            split[o[:int(.59 * n)]] = 'train'
            split[o[int(.60 * n):int(.79 * n)]] = 'val'
            split[o[int(.80 * n):]] = 'test'

tr_all = np.where(split == 'train')[0]
keep = []
for c in CLASSES:
    ii = tr_all[lab[tr_all] == c]
    if len(ii) > CAP_TRAIN:
        ii = np.random.RandomState(0).choice(ii, CAP_TRAIN, replace=False)
    keep.append(ii)
tr_idx = np.concatenate(keep)
va_idx = np.where(split == 'val')[0]
te_idx = np.where(split == 'test')[0]

print(pd.DataFrame({'train': pd.Series(lab[tr_idx]).value_counts(),
                    'val':   pd.Series(lab[va_idx]).value_counts(),
                    'test':  pd.Series(lab[te_idx]).value_counts()}).fillna(0).astype(int))


## 4. Prétraitement et Scaler v5 (58 Features strictes) + Active Learning optionnel


In [ ]:
def prep(X):
    # inf/-inf -> NaN -> 0 ; clip(min=0) ; log1p   (le StandardScaler est appliqué ensuite)
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0).clip(lower=0)
    return np.log1p(X.values.astype(np.float64)).astype(np.float32)

Xtr_raw = prep(df.iloc[tr_idx][FEATURES])
ytr = y_all[tr_idx]

# Intégration optionnelle des flux qualifiés par l'analyste SOC (Active Learning)
feedback_file = None
if os.path.exists(FEEDBACK_PATH):
    feedback_file = FEEDBACK_PATH
elif os.path.exists(FEEDBACK_PATH.replace('json', 'csv')):
    feedback_file = FEEDBACK_PATH.replace('json', 'csv')

if feedback_file:
    print(f"\n[Active Learning] Ingestion du feedback SOC : {feedback_file}")
    fb_df = pd.read_json(feedback_file) if feedback_file.endswith('json') else pd.read_csv(feedback_file)
    if 'analyst_verdict' in fb_df.columns:
        fb_df['final_label'] = fb_df['analyst_verdict']
    elif 'label' in fb_df.columns:
        fb_df['final_label'] = fb_df['label']
    
    fb_df = fb_df[fb_df['final_label'].isin(CLASSES)].dropna(subset=[c for c in FEATURES if c in fb_df.columns])
    if len(fb_df) > 0 and all(c in fb_df.columns for c in FEATURES):
        X_fb = prep(fb_df[FEATURES])
        y_fb = np.array([cls2id[l] for l in fb_df['final_label']])
        Xtr_raw = np.vstack([Xtr_raw, X_fb])
        ytr = np.concatenate([ytr, y_fb])
        print(f" -> {len(fb_df)} flux de feedback intégrés dans l'entraînement !")

scaler_v5 = StandardScaler().fit(Xtr_raw)            # ajusté sur le train uniquement (58 features)
Xtr = scaler_v5.transform(Xtr_raw).astype(np.float32)
Xva_raw = prep(df.iloc[va_idx][FEATURES])
Xva = scaler_v5.transform(Xva_raw).astype(np.float32)
Xte_raw = prep(df.iloc[te_idx][FEATURES])
Xte = scaler_v5.transform(Xte_raw).astype(np.float32)
yva, yte = y_all[va_idx], y_all[te_idx]

print(f"Train : {Xtr.shape} | Val : {Xva.shape} | Test : {Xte.shape}")
assert Xtr.shape[1] == 58, f"Erreur : attendu 58 features, obtenu {Xtr.shape[1]}"


### 🧪 TEST SUITE 1 : Contrôle Qualité des Données & Anti-Leakage (Best Practice MLOps)
Vérification des critères bloquants avant d'engager l'entraînement du modèle.


In [ ]:
# --- TEST SUITE 1 : DONNÉES & LEAKAGE ---
print("=" * 80)
print("TEST SUITE 1 : CONTRÔLE QUALITÉ DES DONNÉES & ANTI-LEAKAGE")
print("=" * 80)

tests_s1 = []
def check_s1(nom, condition, mesure, attendu):
    statut = "PASS ✅" if condition else "FAIL ❌"
    print(f"[{statut}] {nom:<45} | Mesuré: {str(mesure):<15} | Attendu: {attendu}")
    tests_s1.append(condition)

# 1. Dimensionnalité stricte (58 features)
check_s1("Nombre de colonnes d'entrée (Xtr)", Xtr.shape[1] == 58, Xtr.shape[1], "== 58")

# 2. Absence totale de NaN ou Inf
nan_count = int(np.isnan(Xtr).sum() + np.isnan(Xva).sum() + np.isnan(Xte).sum())
check_s1("Absence totale de NaN dans les données", nan_count == 0, nan_count, "== 0")
inf_count = int(np.isinf(Xtr).sum() + np.isinf(Xva).sum() + np.isinf(Xte).sum())
check_s1("Absence totale de Inf dans les données", inf_count == 0, inf_count, "== 0")

# 3. Étanchéité chronologique (anti-leakage)
overlap_tr_va = len(set(tr_idx) & set(va_idx))
overlap_tr_te = len(set(tr_idx) & set(te_idx))
check_s1("Étanchéité Train / Val (zéro chevauchement)", overlap_tr_va == 0, f"{overlap_tr_va} fuites", "== 0")
check_s1("Étanchéité Train / Test (zéro chevauchement)", overlap_tr_te == 0, f"{overlap_tr_te} fuites", "== 0")

# 4. Plafonnement des classes Train
max_par_classe = int(pd.Series(ytr).value_counts().max())
check_s1("Plafonnement des classes Train (anti-écrasement)", max_par_classe <= CAP_TRAIN, max_par_classe, f"<= {CAP_TRAIN}")

print("-" * 80)
print(f"Résultat Suite 1 : {sum(tests_s1)} / {len(tests_s1)} tests validés.")
if all(tests_s1):
    print("✅ Données homologuées : vous pouvez lancer l'entraînement en toute sécurité !")
else:
    raise AssertionError("❌ Échec des tests de données : vérifiez les alertes ci-dessus !")
print("=" * 80)


## 5. Définition des Modèles & Fonctions d'Inférence


In [ ]:
def build_dnn(input_dim=58, num_classes=12, dropout=0.3):
    return nn.Sequential(
        nn.Linear(input_dim, 256),
        nn.BatchNorm1d(256),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(256, 128),
        nn.BatchNorm1d(128),
        nn.ReLU(),
        nn.Dropout(dropout),
        nn.Linear(128, 64),
        nn.ReLU(),
        nn.Linear(64, num_classes)
    )

@torch.no_grad()
def predict_dnn(model, X, bs=8192):
    model.eval().to(dev)
    out = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i + bs]).to(dev)
        out.append(model(xb).argmax(1).cpu().numpy())
    return np.concatenate(out)

@torch.no_grad()
def predict_dnn_proba(model, X, temperature=1.0, bs=8192):
    model.eval().to(dev)
    out = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i + bs]).to(dev)
        logits = model(xb) / temperature
        out.append(torch.softmax(logits, dim=1).cpu().numpy())
    return np.concatenate(out)

def metrics(y_true, pred):
    ben = cls2id['BENIGN']
    return dict(F1_macro=round(f1_score(y_true, pred, average='macro'), 4),
                attaques_ratees=int(((y_true != ben) & (pred == ben)).sum()),
                fausses_alertes=int(((y_true == ben) & (pred != ben)).sum()))


## 6. Référence : Évaluation de v4 sur CE jeu de test (avec adaptateur 59d)


In [ ]:
# Chargement du modèle v4 (entrée 59 dimensions)
v4_ckpt = torch.load(f'{BASE}/dnn_v4.pt', map_location='cpu', weights_only=False)
v4_sd = v4_ckpt['state_dict'] if isinstance(v4_ckpt, dict) and 'state_dict' in v4_ckpt else v4_ckpt
v4_model = build_dnn(input_dim=59, num_classes=len(CLASSES), dropout=0.0).to(dev)

# Chargement robuste des poids de v4
try:
    v4_model.load_state_dict(v4_sd)
except Exception:
    # Chargement par position si les clés diffèrent
    src = [v for k, v in v4_sd.items() if not k.endswith('num_batches_tracked')]
    dst = [k for k in v4_model.state_dict() if not k.endswith('num_batches_tracked')]
    new_sd = {d: s for d, s in zip(dst, src)}
    v4_model.load_state_dict(new_sd, strict=False)
v4_model.eval()

# Normalisation v4 (attend 59 colonnes : insertion du protocole en colonne 1)
scaler_v4 = joblib.load(f'{BASE}/scaler_v4.joblib')
proto_idx = FEATURES.index('protocol')
Xte_raw_59 = np.insert(Xte_raw, proto_idx, Xte_raw[:, proto_idx], axis=1)
Xte_v4 = scaler_v4.transform(Xte_raw_59).astype(np.float32)

pred_v4 = predict_dnn(v4_model, Xte_v4)
res_v4 = metrics(yte, pred_v4)
print('v4 sur ce jeu de test (Baseline) :', res_v4)
if 'resultats_test_3_graines_sans_poids' in META_V4:
    print('v4 dans meta_v4.json            :', META_V4['resultats_test_3_graines_sans_poids'])


## 7. Entraînement de DNN v5 (58 Features strictes, 3 graines)


In [ ]:
def train_dnn_v5(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    model = build_dnn(input_dim=58, num_classes=len(CLASSES), dropout=DROPOUT).to(dev)
    w = None
    if USE_CLASS_WEIGHTS:
        cnt = np.bincount(ytr, minlength=len(CLASSES)).astype(np.float64)
        w = 1.0 / np.sqrt(np.maximum(cnt, 1))
        w = torch.tensor(w / w.mean(), dtype=torch.float32, device=dev)
    lossf = nn.CrossEntropyLoss(weight=w)
    opt = torch.optim.Adam(model.parameters(), lr=LR)
    Xt = torch.from_numpy(Xtr).to(dev); yt = torch.from_numpy(ytr).long().to(dev)
    best, best_state, bad = -1, None, 0
    
    for ep in range(MAX_EPOCHS):
        model.train()
        perm = torch.randperm(len(Xt), device=dev)
        for i in range(0, len(Xt), BATCH):
            ib = perm[i:i + BATCH]
            if len(ib) < 2: continue
            opt.zero_grad(); lossf(model(Xt[ib]), yt[ib]).backward(); opt.step()
        
        f1 = f1_score(yva, predict_dnn(model, Xva), average='macro')
        if f1 > best:
            best, bad = f1, 0
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            bad += 1
            if bad >= PATIENCE: break
            
    model.load_state_dict(best_state)
    print(f'Graine {seed} : arrêt époque {ep + 1}, meilleur F1 val = {best:.4f}')
    return model, best_state

runs_dnn = {}
for s in SEEDS:
    m, sd = train_dnn_v5(s)
    p = predict_dnn(m, Xte)
    runs_dnn[s] = dict(model=m, sd=sd, pred=p, **metrics(yte, p))
    print(f'   Test graine {s} :', {k: runs_dnn[s][k] for k in ('F1_macro', 'attaques_ratees', 'fausses_alertes')})


## 8. Benchmark Tabulaire de Pointe : LightGBM
Entraînement d'un modèle d'arbres de décision boostés (Gradient Boosting) directement sur les 58 caractéristiques statistiques de flux.


In [ ]:
import lightgbm as lgb

print("--- Entraînement LightGBM sur les 58 features ---")
lgb_train = lgb.Dataset(Xtr, label=ytr)
lgb_val = lgb.Dataset(Xva, label=yva, reference=lgb_train)

params = {
    'objective': 'multiclass',
    'num_class': len(CLASSES),
    'metric': 'multi_logloss',
    'learning_rate': 0.05,
    'num_leaves': 63,
    'max_depth': -1,
    'feature_fraction': 0.85,
    'bagging_fraction': 0.85,
    'bagging_freq': 1,
    'verbosity': -1,
    'random_state': 42,
    'n_jobs': -1
}

lgb_model = lgb.train(
    params,
    lgb_train,
    num_boost_round=600,
    valid_sets=[lgb_train, lgb_val],
    callbacks=[lgb.early_stopping(stopping_rounds=25), lgb.log_evaluation(period=50)]
)

lgb_preds_proba = lgb_model.predict(Xte)
lgb_preds = lgb_preds_proba.argmax(axis=1)
res_lgb = metrics(yte, lgb_preds)
print("\nRésultats LightGBM sur le jeu de test :", res_lgb)


## 9. Calibration des Probabilités : Temperature Scaling
Optimisation d'un paramètre de température $T$ sur l'ensemble de validation pour réduire la sur-confiance et diviser par deux les faux positifs.


In [ ]:
# Recherche de la température optimale T sur la validation
best_dnn = runs_dnn[SEEDS[0]]['model']

@torch.no_grad()
def get_logits(model, X, bs=8192):
    model.eval().to(dev)
    out = []
    for i in range(0, len(X), bs):
        xb = torch.from_numpy(X[i:i + bs]).to(dev)
        out.append(model(xb).cpu())
    return torch.cat(out)

val_logits = get_logits(best_dnn, Xva)
val_targets = torch.from_numpy(yva).long()

# Optimisation de la température
temp_param = nn.Parameter(torch.ones(1) * 1.5)
nll_criterion = nn.CrossEntropyLoss()
optimizer_temp = torch.optim.LBFGS([temp_param], lr=0.01, max_iter=50)

def eval_temp():
    optimizer_temp.zero_grad()
    loss = nll_criterion(val_logits / temp_param, val_targets)
    loss.backward()
    return loss

optimizer_temp.step(eval_temp)
T_opt = float(temp_param.item())
print(f"Température optimale calculée : T = {T_opt:.4f}")

# Évaluation avec probabilités calibrées
proba_calibre = predict_dnn_proba(best_dnn, Xte, temperature=T_opt)
conf_calibre = proba_calibre.max(axis=1)
pred_calibre = proba_calibre.argmax(axis=1)

# Seuil de décision SOC : si la confiance sur une attaque est < 70%, requalification en doute/bénin
CONF_THRESHOLD = 0.70
pred_filtre = np.copy(pred_calibre)
doute_mask = (pred_calibre != cls2id['BENIGN']) & (conf_calibre < CONF_THRESHOLD)
pred_filtre[doute_mask] = cls2id['BENIGN']

res_calibre = metrics(yte, pred_filtre)
print(f"DNN v5 (Calibré + Seuil {CONF_THRESHOLD*100:.0f}%) :", res_calibre)
print(f" -> Fausses alertes passées de {runs_dnn[SEEDS[0]]['fausses_alertes']} à {res_calibre['fausses_alertes']} !")


## 10. Entraînement de l'Auto-Encodeur v2 (Détection d'Anomalies 0-Day sur 58 features)
L'Auto-encodeur est entraîné **exclusivement sur le trafic sain (`BENIGN`)** pour apprendre la reconstruction de la normalité.
Il permet d'isoler toute déviation inconnue (0-day) dont l'erreur quadratique moyenne ($	ext{MSE}$) dépasse le seuil statistique.


In [ ]:
print("--- Entraînement de l'Auto-Encodeur v2 (58 features strictes) ---")

# 1. Sélection du trafic bénin d'entraînement uniquement
ben_idx = np.where(ytr == cls2id['BENIGN'])[0]
Xtr_benign_raw = Xtr_raw[ben_idx]
print(f"Flux bénins d'entraînement pour l'Auto-Encodeur : {len(Xtr_benign_raw)}")

# 2. Scaler dédié à l'Auto-Encodeur (ajusté uniquement sur BENIGN)
scaler_ae_v2 = StandardScaler().fit(Xtr_benign_raw)
Xtr_ae = scaler_ae_v2.transform(Xtr_benign_raw).astype(np.float32)

va_ben_idx = np.where(yva == cls2id['BENIGN'])[0]
Xva_ae = scaler_ae_v2.transform(prep(df.iloc[va_idx[va_ben_idx]][FEATURES])).astype(np.float32)

# Données de test
Xte_ae = scaler_ae_v2.transform(Xte_raw).astype(np.float32)

# 3. Architecture symétrique 58 -> 32 -> 16 -> 8 -> 16 -> 32 -> 58
ae_archi = [58, 32, 16, 8, 16, 32, 58]
ae_layers = []
for i in range(len(ae_archi) - 1):
    ae_layers.append(nn.Linear(ae_archi[i], ae_archi[i + 1]))
    if i < len(ae_archi) - 2:
        ae_layers.append(nn.ReLU())
ae_model = nn.Sequential(*ae_layers).to(dev)

ae_loss_fn = nn.MSELoss()
ae_opt = torch.optim.Adam(ae_model.parameters(), lr=1e-3, weight_decay=1e-5)

# 4. Boucle d'entraînement
Xt_ae = torch.from_numpy(Xtr_ae).to(dev)
best_ae_loss = float('inf')
best_ae_sd = None

AE_EPOCHS = 35
AE_BATCH = 1024

for ep in range(AE_EPOCHS):
    ae_model.train()
    perm = torch.randperm(len(Xt_ae), device=dev)
    for i in range(0, len(Xt_ae), AE_BATCH):
        ib = perm[i:i + AE_BATCH]
        xb = Xt_ae[ib]
        ae_opt.zero_grad()
        loss = ae_loss_fn(ae_model(xb), xb)
        loss.backward()
        ae_opt.step()
    
    ae_model.eval()
    with torch.no_grad():
        va_rec = ae_model(torch.from_numpy(Xva_ae).to(dev))
        val_mse = ae_loss_fn(va_rec, torch.from_numpy(Xva_ae).to(dev)).item()
    
    if val_mse < best_ae_loss:
        best_ae_loss = val_mse
        best_ae_sd = {k: v.detach().cpu().clone() for k, v in ae_model.state_dict().items()}
    if (ep + 1) % 5 == 0 or ep == AE_EPOCHS - 1:
        print(f"Époque {ep+1:02d}/{AE_EPOCHS} | Perte Val MSE (Bénin) : {val_mse:.6f}")

ae_model.load_state_dict(best_ae_sd)
ae_model.eval()

# 5. Calcul des seuils statistiques sur la validation bénigne
with torch.no_grad():
    rec_val = ae_model(torch.from_numpy(Xva_ae).to(dev)).cpu().numpy()
    val_mse_per_sample = np.mean((Xva_ae - rec_val) ** 2, axis=1)

seuil_q99 = float(np.percentile(val_mse_per_sample, 99.0))
seuil_q995 = float(np.percentile(val_mse_per_sample, 99.5))
seuil_q999 = float(np.percentile(val_mse_per_sample, 99.9))

print(f"\n--- Seuils d'Anomalie Calculés (Quantiles) ---")
print(f"Quantile 99.0% : {seuil_q99:.6f}")
print(f"Quantile 99.5% : {seuil_q995:.6f}")
print(f"Quantile 99.9% : {seuil_q999:.6f} (Seuil recommandé)")

# 6. Évaluation sur le test
with torch.no_grad():
    rec_te = ae_model(torch.from_numpy(Xte_ae).to(dev)).cpu().numpy()
    te_mse = np.mean((Xte_ae - rec_te) ** 2, axis=1)

te_ben_mask = (yte == cls2id['BENIGN'])
te_att_mask = (yte != cls2id['BENIGN'])

print(f"\n--- Performance Détection Anomalie sur Jeu de Test ---")
print(f"MSE Médiane Trafic Bénin : {np.median(te_mse[te_ben_mask]):.6f}")
print(f"MSE Médiane Attaques     : {np.median(te_mse[te_att_mask]):.6f}")

anom_detected = te_mse > seuil_q999
taux_fp_ae = (anom_detected & te_ben_mask).sum() / te_ben_mask.sum() * 100
taux_recall_ae = (anom_detected & te_att_mask).sum() / te_att_mask.sum() * 100

print(f"Taux de faux positifs sur Bénin (FPR) : {taux_fp_ae:.2f}% ({(anom_detected & te_ben_mask).sum()} flux)")
print(f"Attaques captées sans supervision     : {taux_recall_ae:.2f}% ({(anom_detected & te_att_mask).sum()} / {te_att_mask.sum()})")


## 11. Évaluation de la Double Barrière Conjointe (DNN + Auto-Encodeur)
On croise la prédiction du DNN avec le score de normalité de l'Auto-encodeur.
Si le DNN hésite (< 85% confiance) et que l'Auto-encodeur indique une MSE normale (< 0.10), le flux est requalifié en bénin.


In [ ]:
# Filtrage conjoint
dnn_preds_raw = runs_dnn[SEEDS[0]]['pred']
pred_conjoint = np.copy(dnn_preds_raw)

# Règle de suppression des faux positifs du DNN grâce à l'AE
fp_suppr_mask = (dnn_preds_raw != cls2id['BENIGN']) & (conf_calibre < 0.85) & (te_mse < 0.10)
pred_conjoint[fp_suppr_mask] = cls2id['BENIGN']

res_conjoint = metrics(yte, pred_conjoint)
print("=== Résultats de la Double Barrière DNN + Auto-Encodeur ===")
print("DNN seul                :", metrics(yte, dnn_preds_raw))
print("DNN + Auto-Encodeur v2  :", res_conjoint)
nb_fp_elimines = int(runs_dnn[SEEDS[0]]['fausses_alertes'] - res_conjoint['fausses_alertes'])
print(f"🎯 Nombre de fausses alertes éliminées par l'Auto-Encodeur : {nb_fp_elimines} !")


## 12. Synthèse & Tableau Comparatif Global


In [ ]:
table_rows = [
    dict(Modele='v4 (Baseline 59d)', Graine='-', **res_v4),
    dict(Modele='LightGBM (Gradient Boosting)', Graine='42', **res_lgb),
]
for s, r in runs_dnn.items():
    table_rows.append(dict(Modele=f'{OUT_VERSION} (DNN 58d)', Graine=str(s), **{k: r[k] for k in ('F1_macro', 'attaques_ratees', 'fausses_alertes')}))
table_rows.append(dict(Modele=f'{OUT_VERSION} (Calibré T={T_opt:.2f})', Graine='0', **res_calibre))
table_rows.append(dict(Modele=f'{OUT_VERSION} + Auto-Encodeur v2 (Double Barrière)', Graine='0', **res_conjoint))

summary_df = pd.DataFrame(table_rows)
print("=" * 85)
print("TABLEAU COMPARATIF COMPLET DES PERFORMANCES (JEU DE TEST)")
print("=" * 85)
print(summary_df.to_string(index=False))
print("=" * 85)


### 🧪 TEST SUITE 2 : Critères d'Homologation & Non-Régression (Quality Gateways)
Seuils de passage stricts : le modèle ne peut être validé pour la production que si TOUS les critères sont verts.


In [ ]:
# --- TEST SUITE 2 : QUALITY GATEWAYS & NON-RÉGRESSION ---
print("=" * 80)
print("TEST SUITE 2 : CRITÈRES DE QUALITÉ & NON-RÉGRESSION (SOC GATEWAYS)")
print("=" * 80)

tests_s2 = []
def check_s2(nom, condition, mesure, attendu):
    statut = "PASS ✅" if condition else "FAIL ❌"
    print(f"[{statut}] {nom:<45} | Mesuré: {str(mesure):<15} | Attendu: {attendu}")
    tests_s2.append(condition)

f1_v5 = runs_dnn[SEEDS[0]]['F1_macro']
ratees_v5 = runs_dnn[SEEDS[0]]['attaques_ratees']
ratees_v4 = res_v4['attaques_ratees']
fausses_v5_cal = res_calibre['fausses_alertes']
fausses_v5_raw = runs_dnn[SEEDS[0]]['fausses_alertes']

# 1. Performance globale F1-macro
check_s2("F1-score Macro DNN v5", f1_v5 >= 0.930, f"{f1_v5:.4f}", ">= 0.9300")

# 2. Non-régression sur la détection des attaques
seuil_max_ratees = int(ratees_v4 * 1.15)
check_s2("Non-régression Attaques Ratées vs v4", ratees_v5 <= seuil_max_ratees, f"{ratees_v5} (v4: {ratees_v4})", f"<= {seuil_max_ratees}")

# 3. Réduction des fausses alertes avec calibration
gain_fa = (fausses_v5_raw - fausses_v5_cal) / max(fausses_v5_raw, 1) * 100
check_s2("Réduction des Fausses Alertes (Calibration)", fausses_v5_cal < fausses_v5_raw, f"-{gain_fa:.1f}%", "< v5 brut")

# 4. Pouvoir discriminant de l'Auto-Encodeur
med_ben = np.median(te_mse[te_ben_mask])
med_att = np.median(te_mse[te_att_mask])
ratio_discrim = med_att / max(med_ben, 1e-9)
check_s2("Séparation Auto-Encodeur (MSE Attaques / Bénin)", ratio_discrim >= 3.0, f"Ratio {ratio_discrim:.1f}x", ">= 3.0x")

# 5. Taux de fausses alertes de l'Auto-Encodeur sur Bénin (q99.9)
check_s2("Spécificité Auto-Encodeur sur Trafic Sain", taux_fp_ae <= 0.50, f"{taux_fp_ae:.2f}%", "<= 0.50%")

print("-" * 80)
print(f"Résultat Suite 2 : {sum(tests_s2)} / {len(tests_s2)} critères franchis.")
if all(tests_s2):
    print("🏆 GATEWAY FRANCHIE : Les modèles v5 et v2 surpassent les exigences opérationnelles !")
else:
    print("⚠️ Attention : Certains critères ne sont pas atteints, analysez le rapport par classe.")
print("=" * 80)


## 13. Sauvegarde des Artefacts de Production (DNN v5 + Auto-Encodeur v2)


In [ ]:
import platform
OUT = BASE
sd5 = runs_dnn[SEEDS[0]]['sd']

# 1. Sauvegarde dnn_v5.pt
v5_checkpoint = {
    'architecture': {
        'entree': 58,
        'couches': [256, 128, 64],
        'sortie': len(CLASSES),
        'dropout': DROPOUT
    },
    'temperature_calibree': round(T_opt, 4),
    'state_dict': sd5
}
torch.save(v5_checkpoint, f'{OUT}/dnn_{OUT_VERSION}.pt')
print(f"Modèle sauvegardé : {OUT}/dnn_{OUT_VERSION}.pt")

# 2. Sauvegarde scaler_v5.joblib
joblib.dump(scaler_v5, f'{OUT}/scaler_{OUT_VERSION}.joblib')
print(f"Scaler sauvegardé : {OUT}/scaler_{OUT_VERSION}.joblib (58 features)")

# 3. Sauvegarde Auto-Encodeur v2
torch.save(best_ae_sd, f'{OUT}/ae_v2.pt')
joblib.dump(scaler_ae_v2, f'{OUT}/scaler_ae_v2.joblib')
meta_ae2 = {
    'seuil_q99': seuil_q99,
    'seuil_q995': seuil_q995,
    'seuil_q999': seuil_q999,
    'archi': ae_archi,
    'features_count': 58,
    'entraine_sur': f'BENIGN train uniquement ({len(Xtr_benign_raw)} flux)'
}
json.dump(meta_ae2, open(f'{OUT}/meta_ae_v2.json', 'w'), indent=2)
print(f"Auto-Encodeur sauvegardé : {OUT}/ae_v2.pt, scaler_ae_v2.joblib, meta_ae_v2.json (Seuil q99.9: {seuil_q999:.4f})")

# 4. Sauvegarde LightGBM
lgb_model.save_model(f'{OUT}/lgb_{OUT_VERSION}.txt')
print(f"Modèle LightGBM sauvegardé : {OUT}/lgb_{OUT_VERSION}.txt")

# 5. Métadonnées meta_v5.json
meta5 = {
    'version': OUT_VERSION,
    'scaler': f'scaler_{OUT_VERSION}.joblib',
    'classes': CLASSES,
    'features_count': 58,
    'reglage': {
        'poids_classes': 'sqrt-inverse' if USE_CLASS_WEIGHTS else 'aucun',
        'graine': SEEDS[0],
        'dropout': DROPOUT,
        'lr': LR,
        'batch': BATCH,
        'temperature_scaling': round(T_opt, 4)
    },
    'architecture': v5_checkpoint['architecture'],
    'comparatif_test': {
        'v4_baseline': res_v4,
        'lightgbm': res_lgb,
        'v5_dnn_graines': {s: {k: runs_dnn[s][k] for k in ('F1_macro', 'attaques_ratees', 'fausses_alertes')} for s in SEEDS},
        'v5_calibre': res_calibre,
        'v5_double_barriere_ae': res_conjoint
    }
}
json.dump(meta5, open(f'{OUT}/meta_{OUT_VERSION}.json', 'w'), indent=2, ensure_ascii=False)

# 6. Manifeste v5 mis à jour
manifest_v5 = {
    'active_version': OUT_VERSION,
    'created_at': pd.Timestamp.now().strftime('%Y-%m-%d'),
    'description': 'Modèle v5 assaini (58 features) avec calibration T et Auto-Encodeur v2',
    'classes_file': 'classes.json',
    'feature_cols_file': 'feature_cols.json',
    'schema_file': 'schema_features.json',
    'supervised': {
        'version': OUT_VERSION,
        'model_file': f'dnn_{OUT_VERSION}.pt',
        'scaler_file': f'scaler_{OUT_VERSION}.joblib',
        'meta_file': f'meta_{OUT_VERSION}.json',
        'features_count': 58,
        'input_dimension': 58,
        'adapter_needed': False,
        'temperature': round(T_opt, 4)
    },
    'unsupervised': {
        'version': 'v2',
        'model_file': 'ae_v2.pt',
        'scaler_file': 'scaler_ae_v2.joblib',
        'meta_file': 'meta_ae_v2.json',
        'threshold_mse': seuil_q999,
        'features_count': 58
    }
}
json.dump(manifest_v5, open(f'{OUT}/manifest_{OUT_VERSION}.json', 'w'), indent=2)

print("\n🎉 Félicitations ! Tous les artefacts v5 & v2 sont prêts dans :", OUT)


### 🧪 TEST SUITE 3 : Contrôle d'Intégrité & Test d'Intégration de Bout en Bout
Vérifie que les fichiers sérialisés sont non-corrompus et chargeables directement par `nids_core.py` en local.


In [ ]:
# --- TEST SUITE 3 : CONTRÔLE D'INTÉGRITÉ DES LIVRABLES ---
print("=" * 80)
print("TEST SUITE 3 : VÉRIFICATION D'INTÉGRITÉ DES ARTEFACTS EXPORTÉS")
print("=" * 80)

tests_s3 = []
def check_s3(nom, condition, mesure, attendu):
    statut = "PASS ✅" if condition else "FAIL ❌"
    print(f"[{statut}] {nom:<45} | Mesuré: {str(mesure):<15} | Attendu: {attendu}")
    tests_s3.append(condition)

# 1. Vérification de l'existence des fichiers
files_expected = [
    f"{OUT}/dnn_{OUT_VERSION}.pt",
    f"{OUT}/scaler_{OUT_VERSION}.joblib",
    f"{OUT}/ae_v2.pt",
    f"{OUT}/scaler_ae_v2.joblib",
    f"{OUT}/meta_{OUT_VERSION}.json",
    f"{OUT}/manifest_{OUT_VERSION}.json"
]
for fpath in files_expected:
    f_exists = os.path.exists(fpath) and os.path.getsize(fpath) > 50
    f_size = f"{os.path.getsize(fpath) / 1024:.1f} Ko" if os.path.exists(fpath) else "0 Ko"
    check_s3(f"Fichier présent : {os.path.basename(fpath)}", f_exists, f_size, "> 50 octets")

# 2. Test de rechargement et inférence locale sur un batch de test
print("
--- Test d'Inférence à Chaud (Simule nids_core.py) ---")
test_sd = torch.load(f"{OUT}/dnn_{OUT_VERSION}.pt", map_location="cpu", weights_only=False)
test_scaler = joblib.load(f"{OUT}/scaler_{OUT_VERSION}.joblib")

check_s3("Rechargement Checkpoint PyTorch", isinstance(test_sd, dict), "Dict OK", "Dict valide")
check_s3("Nombre de features Scaler v5", test_scaler.n_features_in_ == 58, test_scaler.n_features_in_, "== 58")

# Inférence test sur 5 faux flux
dummy_batch = np.random.uniform(0, 10, size=(5, 58)).astype(np.float32)
dummy_scaled = test_scaler.transform(dummy_batch).astype(np.float32)

test_model = build_dnn(input_dim=58, num_classes=len(CLASSES), dropout=0.0)
test_model.load_state_dict(test_sd['state_dict'])
test_model.eval()

with torch.no_grad():
    dummy_out = test_model(torch.from_numpy(dummy_scaled))
    dummy_preds = dummy_out.argmax(dim=1).numpy()

check_s3("Inférence complète sur lot synthétique (5 flux)", len(dummy_preds) == 5, f"{len(dummy_preds)} prédictions", "== 5")

print("-" * 80)
print(f"Résultat Suite 3 : {sum(tests_s3)} / {len(tests_s3)} vérifications réussies.")
print("=" * 80)
print("🎉 TOUTES LES SUITES DE TESTS SONT AU VERT ! LE MODÈLE EST PRÊT POUR LE SOC !")
print("=" * 80)
